# 🥚 Pelatihan YOLOv8-seg (Instance Segmentation) - Cacat Cangkang Telur
Notebook ini melatih model **YOLOv8-seg** untuk melakukan **segmentasi piksel cacat retak (*cracked*) dan noda kotoran (*dirt*)** pada cangkang telur ayam sesuai acuan standar SNI 3926:2023.

- **Arsitektur:** `yolov8n-seg.pt` (Instance Segmentation)
- **Target Cacat:** Retak (cracked) dan Kotoran (dirt)
- **Telur Mulus (good):** Background / Negative Sample (tanpa polygon mask)
- **Metrik:** Box mAP & Mask mAP (evaluasi presisi piksel)

In [ ]:
# 1. Cek Akselerasi GPU Colab & Install Ultralytics
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# 2. Persiapan Dataset Segmentasi
# Cara A: Jika mengunggah file zip dataset ke panel Files (kiri Colab)
import os
import zipfile

# Cari file zip yang diupload (misal: dataset_seg.zip atau roboflow.zip)
zip_candidates = [f for f in os.listdir('/content') if f.endswith('.zip')]
extract_dir = '/content/dataset'

if zip_candidates:
    zip_path = os.path.join('/content', zip_candidates[0])
    print(f'📦 Mengekstrak dataset: {zip_path} ke {extract_dir}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print('✅ Dataset berhasil diekstrak!')
else:
    print('ℹ️ Belum ada file .zip di /content. Silakan drag-and-drop file dataset zip Anda ke panel Files (kiri).')

# Cara B (Alternatif): Jika mendownload langsung via Roboflow API
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='MASUKKAN_ROBOFLOW_API_KEY_ANDA')
# project = rf.workspace('WORKSPACE_ANDA').project('PROJECT_ANDA')
# version = project.version(1)
# dataset = version.download('yolov8')

In [ ]:
# 3. Konfigurasi data.yaml (Path Dataset Colab)
import os
import yaml

# Cari file data.yaml di dalam folder ekstraksi
found_yaml = None
for root, dirs, files in os.walk('/content'):
    if 'data.yaml' in files:
        found_yaml = os.path.join(root, 'data.yaml')
        break

if found_yaml:
    with open(found_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    # Set absolute root path ke direktori tempat data.yaml berada
    cfg['path'] = os.path.dirname(found_yaml)
    with open(found_yaml, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    data_yaml_path = found_yaml
    print(f'✅ Konfigurasi data.yaml ditemukan di: {data_yaml_path}')
    print(yaml.dump(cfg))
else:
    data_yaml_path = '/content/dataset/data.yaml'
    default_cfg = {
        'path': '/content/dataset',
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': 2,
        'names': ['cracked', 'dirt']
    }
    os.makedirs('/content/dataset', exist_ok=True)
    with open(data_yaml_path, 'w') as f:
        yaml.dump(default_cfg, f, default_flow_style=False)
    print(f'✅ data.yaml baru berhasil dibuat di: {data_yaml_path}')
    print(yaml.dump(default_cfg))

In [ ]:
# 4. Mulai Pelatihan Model YOLOv8-seg (Instance Segmentation)
from ultralytics import YOLO

# Memuat arsitektur pre-trained YOLOv8 Nano Segmentation
model = YOLO('yolov8n-seg.pt')

print('🚀 Memulai pelatihan YOLOv8-seg pada GPU Colab...')
results = model.train(
    data=data_yaml_path,
    epochs=50,          # Jumlah epoch (bisa disesuaikan: 50 - 100)
    imgsz=640,
    batch=16,
    device=0,           # Menggunakan T4 GPU
    project='/content/runs',
    name='egg_defect_yolov8_seg',
    exist_ok=True,
    patience=15,
    save=True,
    plots=True,
    verbose=True
)

In [ ]:
# 5. Evaluasi Performa Model (Box mAP & Mask mAP)
metrics = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='eval_test_seg'
)

print('\n' + '='*55)
print('📊 HASIL EVALUASI SEGMENTASI (TEST SET):')
print('='*55)
print(f'📦 Bounding Box mAP50        : {metrics.box.map50:.4f}')
print(f'📦 Bounding Box mAP50-95     : {metrics.box.map:.4f}')
print(f'🎭 Segmentation Mask mAP50   : {metrics.seg.map50:.4f}')
print(f'🎭 Segmentation Mask mAP50-95: {metrics.seg.map:.4f}')
print('='*55)

In [ ]:
# 6. Visualisasi Hasil Segmentasi (Masking Cacat Cangkang)
import os
import glob
from IPython.display import Image, display

# Menampilkan visualisasi hasil prediksi segmentasi
val_preds = glob.glob('/content/runs/egg_defect_yolov8_seg/val_batch*_pred.jpg')
charts = [
    '/content/runs/egg_defect_yolov8_seg/results.png',
    '/content/runs/egg_defect_yolov8_seg/confusion_matrix.png'
]

for path in charts + val_preds[:2]:
    if os.path.exists(path):
        print(f'\n=== {os.path.basename(path)} ===')
        display(Image(path, width=720))

In [ ]:
# 7. Unduh Bobot Model Segmentasi Terbaik (best.pt)
from google.colab import files

best_seg_weight = '/content/runs/egg_defect_yolov8_seg/weights/best.pt'
if os.path.exists(best_seg_weight):
    print('⬇️ Mengunduh file best.pt ke komputer Anda...')
    files.download(best_seg_weight)
else:
    print('⚠️ File best.pt belum ditemukan. Pastikan proses training sudah selesai.')

In [ ]:
# 8. Unduh Seluruh Grafik, Matriks Konfusi & Metrik Evaluasi (Format .ZIP)
import os
import shutil
from google.colab import files

run_dir = '/content/runs/egg_defect_yolov8_seg'
zip_output = '/content/hasil_grafik_dan_matriks_yolov8seg.zip'

if os.path.exists(run_dir):
    print('📦 Mengemas seluruh grafik (results.png, confusion_matrix, PR-curve, visualisasi mask, dll)...')
    shutil.make_archive('/content/hasil_grafik_dan_matriks_yolov8seg', 'zip', run_dir)
    print('⬇️ Mengunduh file zip ke komputer Anda...')
    files.download(zip_output)
else:
    print(f'⚠️ Folder {run_dir} tidak ditemukan. Pastikan proses training sudah selesai.')